# Benchmark Qwen3 

| Model | Model ID (Mặc định) |
|-------| ---------------------|
| **Qwen3-4B** | `qwen3-4b` |
| **Qwen3-14B** | `qwen-3-14b` |
| **Qwen3-32B** | `qwen-3-32b` |
| **Qwen3-235B-A22B-Thinking** | `qwen3-235b-a22b-thinking-2507` |

Kết quả của mỗi model sẽ được lưu thành các file JSON riêng biệt ở thư mục `outputs/qwen3`.

## 1. Cài đặt và Import

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.notebook import tqdm
import pandas as pd

## 2. Thiết lập Môi trường & Khởi tạo API Clients

Cần chuẩn bị sẵn file `.env` ở thư mục dự án chứa `OPENROUTER_API_KEY`.

In [ ]:
# Xác định thư mục root của dự án
PROJECT_DIR = Path.cwd().parent
os.chdir(PROJECT_DIR)

load_dotenv(PROJECT_DIR / ".env")

DATA_FILE = PROJECT_DIR / "data" / "bigdata_10_questions.json"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "qwen3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Khởi tạo OpenRouter Client
or_api_key = os.getenv("OPENROUTER_API_KEY")
or_base_url = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")

if not or_api_key:
    print("⚠️ Không tìm thấy OPENROUTER_API_KEY trong file .env!")
    or_client = None
else:
    or_client = OpenAI(base_url=or_base_url, api_key=or_api_key)

# Khởi tạo LM Studio Client (Local)
lm_base_url = os.getenv("LM_STUDIO_BASE_URL", "http://localhost:1234/v1")
lm_client = OpenAI(base_url=lm_base_url, api_key="lm-studio")

print("✅ Môi trường đã sẵn sàng.")
print("Project Dir:", PROJECT_DIR)

## 3. Các hàm hỗ trợ sinh văn bản (Helper Functions)

Các hàm này sẽ gọi API, xử lý prompt, bóc tách suy luận (thinking) và tự động lưu kết quả ra file json.

In [ ]:
SYSTEM_PROMPT = (
    "You are a senior Big Data engineer and Spark expert. "
    "Answer accurately, practically, and follow the user's instructions. "
    "For code tasks, provide correct and production-aware code."
)

def parse_generation_params(param_text: str | None) -> tuple[float, int]:
    temp, max_tokens = 0.6, 1500
    if param_text:
        temp_match = re.search(r"temperature\s*=\s*([0-9.]+)", param_text)
        token_match = re.search(r"max_tokens\s*=\s*(\d+)", param_text)
        if temp_match: temp = float(temp_match.group(1))
        if token_match: max_tokens = int(token_match.group(1))
    return temp, max_tokens

def parse_thinking_content(response_text: str) -> tuple[str, str]:
    if not response_text: return "", ""
    think_match = re.search(r"<think>(.*?)</think>", response_text, re.DOTALL)
    if think_match:
        return think_match.group(1).strip(), response_text[think_match.end():].strip()
    return "", response_text.strip()

def run_and_save_benchmark(
    client: OpenAI, 
    model_id: str, 
    model_name: str, 
    provider: str,
    limit: int = 0,
    sleep_sec: float = 2.0
):
    if not client:
        print(f"❌ Client cho {model_name} chưa được thiết lập. Bỏ qua.")
        return
    
    with open(DATA_FILE, "r", encoding="utf-8") as f:
        samples = json.load(f)
    if limit > 0: samples = samples[:limit]
        
    results = []
    print(f"\n🚀 Đang sinh dữ liệu cho: {model_name} [{model_id}] qua {provider}")
    
    for sample in tqdm(samples, desc=model_name):
        temp, max_tokens = parse_generation_params(sample.get("recommended_generation_params"))
        started_at = datetime.now(timezone.utc).isoformat()
        
        try:
            start = time.perf_counter()
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": sample["prompt"]},
                ],
                temperature=temp,
                max_tokens=max_tokens,
            )
            latency_s = time.perf_counter() - start
            full_response = response.choices[0].message.content or ""
            thinking, answer = parse_thinking_content(full_response)
            error = None
        except Exception as e:
            full_response, thinking, answer, latency_s, error = None, None, None, None, str(e)
            print(f"  ⚠️ Lỗi tại {sample.get('sample_id')}: {error}")

        results.append({
            "sample_id": sample.get("sample_id"),
            "model_name": model_name,
            "model_id": model_id,
            "provider": provider,
            "prompt": sample.get("prompt"),
            "model_output": answer,
            "thinking_content": thinking,
            "full_response": full_response,
            "error": error,
            "latency_s": round(latency_s, 3) if latency_s else None,
            "metadata": {"started_at": started_at, "ended_at": datetime.now(timezone.utc).isoformat()},
        })
        time.sleep(sleep_sec)

    # Lưu file
    fname = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_name).strip("_") + "_outputs.json"
    output_path = OUTPUT_DIR / fname
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"✅ Đã lưu kết quả tại: {output_path}")

## 4. Chạy Qwen3-4B (LM Studio)

In [ ]:
run_and_save_benchmark(
    client=lm_client,
    model_id="qwen3-4b",
    model_name="Qwen3-4B",
    provider="lm_studio",
    limit=0,
)

## 5. Chạy Qwen3-14B (OpenRouter)

In [ ]:
run_and_save_benchmark(
    client=or_client,
    model_id="qwen/qwen-3-14b",
    model_name="Qwen3-14B",
    provider="openrouter",
    limit=0,
)

## 6. Chạy Qwen3-32B (OpenRouter)

In [ ]:
run_and_save_benchmark(
    client=or_client,
    model_id="qwen/qwen-3-32b",
    model_name="Qwen3-32B",
    provider="openrouter",
    limit=0,
)

## 7. Chạy Qwen3-235B-A22B-Thinking (OpenRouter)

In [ ]:
run_and_save_benchmark(
    client=or_client,
    model_id="qwen/qwen3-235b-a22b-thinking-2507",
    model_name="Qwen3-235B-A22B-Thinking-2507",
    provider="openrouter",
    limit=0,
)

## 8. Xem thử dữ liệu đã sinh ra

In [ ]:
# Đọc ngẫu nhiên vài dòng kết quả từ các model đã chạy
all_rows = []
for path in OUTPUT_DIR.glob("*_outputs.json"):
    try:
        all_rows.extend(json.loads(path.read_text(encoding="utf-8")))
    except Exception:
        pass

if all_rows:
    df = pd.DataFrame(all_rows)
    display(df[["sample_id", "model_name", "provider", "latency_s", "error"]].head(15))
else:
    print("Chưa có dữ liệu nào!")